# Lab 1: Numeric Computation in Python — Understanding NumPy Under the Hood

## 🎯 Learning Objectives

By the end of this lab, you will understand:
- How Python's magic methods enable operator overloading
- How NumPy implements numeric operations internally
- The difference between element-wise and matrix operations
- How to build a basic numeric computation library from scratch

**Why This Matters:** Deep learning frameworks like PyTorch and TensorFlow are built on these same principles. Understanding the fundamentals will help you debug issues and build custom operations later.

**Note:** If running in Google Colab, all packages are pre-installed. No setup needed!

## Part 1: Python Magic Methods — The Foundation of Operator Overloading

### What Are Magic Methods?

Magic methods (also called **dunder methods**, short for "double underscore") are special methods that Python calls when you use operators or built-in functions on objects.

### Example: How `+` Really Works

When you write `a + b`, Python translates this to `a.__add__(b)`. Let's see this in action:

In [ ]:
# Regular addition
a = 5
b = 3
print(f"a + b = {a + b}")          # Output: 8
print(f"a.__add__(b) = {a.__add__(b)}")  # Output: 8 (same thing!)

# Even built-in types use magic methods!
print(f"\nType of 5: {type(5)}")
print(f"5 has __add__ method: {hasattr(5, '__add__')}")

### Why This Matters for Deep Learning

When you write `c = a + b` in NumPy or PyTorch:
```python
import numpy as np
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])
c = a + b  # This calls a.__add__(b)
```

The library implements custom `__add__` that:
1. Performs element-wise addition
2. Validates shapes match
3. Uses optimized C code for speed

Later in this course, we'll use the same technique to build a **computation graph** that tracks operations for automatic differentiation!

### Key Magic Methods for Numeric Computation

| Magic Method | Operator | Called When | Example |
|--------------|----------|-------------|----------|
| `__add__(self, other)` | `+` | Addition | `a + b` |
| `__sub__(self, other)` | `-` | Subtraction | `a - b` |
| `__mul__(self, other)` | `*` | Multiplication | `a * b` |
| `__truediv__(self, other)` | `/` | Division | `a / b` |
| `__matmul__(self, other)` | `@` | Matrix multiply | `a @ b` |
| `__repr__(self)` | `print()` | String representation | `print(a)` |
| `__getitem__(self, key)` | `[]` | Indexing | `a[0]` |

**Note:** There are also "reverse" versions (`__radd__`, `__rmul__`, etc.) for when the left operand doesn't support the operation.

## Part 2: Building a Simple Numeric Array Class

Let's build our own `Array` class to understand how NumPy works internally.

### Design Goals:
1. Store data in a Python list
2. Support element-wise operations (+, -, *, /)
3. Validate shapes match for operations
4. Pretty printing

### Demo: A Complete Implementation

Study this implementation carefully — you'll build something similar in the exercise!

In [ ]:
class SimpleArray:
    """A simple 1D array class demonstrating operator overloading."""
    
    def __init__(self, data):
        """Initialize with a Python list."""
        self.data = list(data)  # Store as list
        self.size = len(data)   # Track size
    
    def __add__(self, other):
        """Element-wise addition: self + other"""
        # Validate sizes match
        if self.size != other.size:
            raise ValueError(f"Size mismatch: {self.size} vs {other.size}")
        
        # Perform element-wise addition
        result = [self.data[i] + other.data[i] for i in range(self.size)]
        return SimpleArray(result)
    
    def __mul__(self, other):
        """Element-wise multiplication: self * other"""
        if self.size != other.size:
            raise ValueError(f"Size mismatch: {self.size} vs {other.size}")
        
        result = [self.data[i] * other.data[i] for i in range(self.size)]
        return SimpleArray(result)
    
    def __repr__(self):
        """Pretty printing."""
        return f"SimpleArray({self.data})"

# Test it!
a = SimpleArray([1, 2, 3])
b = SimpleArray([4, 5, 6])

print(f"a = {a}")
print(f"b = {b}")
print(f"a + b = {a + b}")  # Calls a.__add__(b)
print(f"a * b = {a * b}")  # Calls a.__mul__(b)

### How This Compares to NumPy

Let's compare our implementation with NumPy:

In [ ]:
import numpy as np

# NumPy arrays
a_np = np.array([1, 2, 3])
b_np = np.array([4, 5, 6])

print("NumPy:")
print(f"a_np + b_np = {a_np + b_np}")  # Same result!
print(f"a_np * b_np = {a_np * b_np}")  # Same result!

print("\nKey Differences:")
print(f"1. NumPy uses C arrays (fast): {type(a_np.data)}")
print(f"2. Our class uses Python lists (slow): {type(a.data)}")
print(f"3. NumPy supports broadcasting (flexible shapes)")
print(f"4. NumPy has hundreds of optimized operations")

## Part 3: Matrix Multiplication — The Most Important Operation

### Why Matrix Multiplication Matters

Matrix multiplication is the **core operation** in deep learning:
- Forward pass: `output = input @ weights`
- Backward pass: Compute gradients via matrix products

Understanding it deeply is crucial!

### Element-wise vs Matrix Multiplication

**Element-wise multiplication** (what we did above with `*`):
```
[1, 2, 3] * [4, 5, 6] = [1*4, 2*5, 3*6] = [4, 10, 18]
```

**Matrix multiplication** (what `@` does in NumPy):
```
A @ B where A is shape (m, n) and B is shape (n, p)
Result is shape (m, p)
```

### How Matrix Multiplication Works

For matrices A (2×3) and B (3×2):

```
A = [[1, 2, 3],      B = [[7, 8],
     [4, 5, 6]]           [9, 10],
                          [11, 12]]

C = A @ B  (result will be 2×2)

C[0,0] = A[0,:] · B[:,0] = 1*7 + 2*9 + 3*11 = 7 + 18 + 33 = 58
C[0,1] = A[0,:] · B[:,1] = 1*8 + 2*10 + 3*12 = 8 + 20 + 36 = 64
C[1,0] = A[1,:] · B[:,0] = 4*7 + 5*9 + 6*11 = 28 + 45 + 66 = 139
C[1,1] = A[1,:] · B[:,1] = 4*8 + 5*10 + 6*12 = 32 + 50 + 72 = 154

C = [[58, 64],
     [139, 154]]
```

**Key Rules:**
1. Inner dimensions must match: `(m, n) @ (n, p)` ✓
2. Result shape: `(m, p)`
3. Each element is a dot product of a row and column

In [ ]:
# Demo: Matrix multiplication step-by-step
import numpy as np

A = np.array([[1, 2, 3],
              [4, 5, 6]])

B = np.array([[7, 8],
              [9, 10],
              [11, 12]])

print("Matrix A (2×3):")
print(A)
print("\nMatrix B (3×2):")
print(B)

# NumPy matrix multiplication
C = A @ B
print("\nResult C = A @ B (2×2):")
print(C)

# Manual verification of first element
print("\nManual calculation of C[0,0]:")
print(f"A[0,:] = {A[0,:]}")
print(f"B[:,0] = {B[:,0]}")
print(f"Dot product = {A[0,0]*B[0,0]} + {A[0,1]*B[1,0]} + {A[0,2]*B[2,0]} = {A[0,0]*B[0,0] + A[0,1]*B[1,0] + A[0,2]*B[2,0]}")
print(f"Matches C[0,0] = {C[0,0]} ✓")

### Visualizing the Process

Think of matrix multiplication as:
1. Take a **row** from the left matrix
2. Take a **column** from the right matrix
3. Multiply corresponding elements
4. Sum them up
5. That's one element in the result!
6. Repeat for all row-column combinations

```
      [col0  col1]
        ↓     ↓
[row0]  •     •    ← Each • is a dot product
[row1]  •     •
```

## Exercise: Build Your Own Array Class

### Goal

Implement an `Array` class that supports:
1. Element-wise addition (`+`)
2. Element-wise subtraction (`-`)
3. Element-wise multiplication (`*`)
4. Matrix multiplication (`@`)
5. Pretty printing

### Requirements

- Store data as a 2D list: `[[row0], [row1], ...]`
- Track shape: `(rows, cols)`
- Validate shapes for operations
- **NO NUMPY ALLOWED** in your implementation (you need to understand the internals first!)

### Starter Code

Fill in the TODOs below:

In [ ]:
class Array:
    """A 2D array class for numeric computation (no NumPy allowed!)."""
    
    def __init__(self, data):
        """Initialize array from 2D list."""
        self.data = data
        self.rows = len(data)
        self.cols = len(data[0]) if data else 0
        self.shape = (self.rows, self.cols)
    
    def __add__(self, other):
        """Element-wise addition."""
        if self.shape != other.shape:
            raise ValueError(f"Shape mismatch: {self.shape} vs {other.shape}")
        
        result = [
            [self.data[i][j] + other.data[i][j] for j in range(self.cols)]
            for i in range(self.rows)
        ]
        return Array(result)
    
    def __sub__(self, other):
        """Element-wise subtraction."""
        if self.shape != other.shape:
            raise ValueError(f"Shape mismatch: {self.shape} vs {other.shape}")
        
        result = [
            [self.data[i][j] - other.data[i][j] for j in range(self.cols)]
            for i in range(self.rows)
        ]
        return Array(result)
    
    def __mul__(self, other):
        """Element-wise multiplication."""
        if self.shape != other.shape:
            raise ValueError(f"Shape mismatch: {self.shape} vs {other.shape}")
        
        result = [
            [self.data[i][j] * other.data[i][j] for j in range(self.cols)]
            for i in range(self.rows)
        ]
        return Array(result)
    
    def __matmul__(self, other):
        """Matrix multiplication: self @ other"""
        if self.cols != other.rows:
            raise ValueError(f"Inner dimensions don't match: {self.shape} @ {other.shape}")
        
        result = [
            [
                sum(self.data[i][k] * other.data[k][j] for k in range(self.cols))
                for j in range(other.cols)
            ]
            for i in range(self.rows)
        ]
        return Array(result)
    
    def __repr__(self):
        """Pretty printing."""
        return f"Array({self.data})"

# Tests will validate the implementation

### Test Your Implementation

Run these tests to validate your Array class:

In [ ]:
print("=" * 60)
print("Test 1: Element-wise Addition")
print("=" * 60)

A = Array([[1, 2], [3, 4]])
B = Array([[5, 6], [7, 8]])
C = A + B

print(f"A = {A}")
print(f"B = {B}")
print(f"C = A + B = {C}")

expected = [[6, 8], [10, 12]]
assert C.data == expected, f"Expected {expected}, got {C.data}"
print("✓ Addition: PASS\n")

In [ ]:
print("=" * 60)
print("Test 2: Element-wise Subtraction")
print("=" * 60)

D = B - A
print(f"D = B - A = {D}")

expected = [[4, 4], [4, 4]]
assert D.data == expected, f"Expected {expected}, got {D.data}"
print("✓ Subtraction: PASS\n")

In [ ]:
print("=" * 60)
print("Test 3: Element-wise Multiplication")
print("=" * 60)

E = Array([[2, 2], [2, 2]])
F = A * E
print(f"A = {A}")
print(f"E = {E}")
print(f"F = A * E = {F}")

expected = [[2, 4], [6, 8]]
assert F.data == expected, f"Expected {expected}, got {F.data}"
print("✓ Element-wise multiplication: PASS\n")

In [ ]:
print("=" * 60)
print("Test 4: Matrix Multiplication")
print("=" * 60)

# 2x2 @ 2x2 = 2x2
G = A @ B
print(f"A = {A}")
print(f"B = {B}")
print(f"G = A @ B = {G}")

# Manual calculation:
# G[0][0] = 1*5 + 2*7 = 5 + 14 = 19
# G[0][1] = 1*6 + 2*8 = 6 + 16 = 22
# G[1][0] = 3*5 + 4*7 = 15 + 28 = 43
# G[1][1] = 3*6 + 4*8 = 18 + 32 = 50
expected = [[19, 22], [43, 50]]
assert G.data == expected, f"Expected {expected}, got {G.data}"
print("✓ Matrix multiplication: PASS\n")

In [ ]:
print("=" * 60)
print("Test 5: Non-square Matrix Multiplication")
print("=" * 60)

# 2x3 @ 3x2 = 2x2
H = Array([[1, 2, 3], [4, 5, 6]])
I = Array([[7, 8], [9, 10], [11, 12]])
J = H @ I

print(f"H (2x3) = {H}")
print(f"I (3x2) = {I}")
print(f"J = H @ I (2x2) = {J}")

# Manual: 
# J[0][0] = 1*7 + 2*9 + 3*11 = 7 + 18 + 33 = 58
# J[0][1] = 1*8 + 2*10 + 3*12 = 8 + 20 + 36 = 64
# J[1][0] = 4*7 + 5*9 + 6*11 = 28 + 45 + 66 = 139
# J[1][1] = 4*8 + 5*10 + 6*12 = 32 + 50 + 72 = 154
expected = [[58, 64], [139, 154]]
assert J.data == expected, f"Expected {expected}, got {J.data}"
print("✓ Non-square matmul: PASS\n")

In [ ]:
print("=" * 60)
print("🎉 All Tests Passed!")
print("=" * 60)
print("\nYou've successfully implemented a basic numeric computation library!")
print("This is essentially how NumPy works under the hood (but much faster).")

## Bonus Challenge: Compare with NumPy

Now that you understand the internals, let's compare your implementation with NumPy:

In [ ]:
import numpy as np

# Your implementation
a_custom = Array([[1, 2, 3], [4, 5, 6]])
b_custom = Array([[7, 8], [9, 10], [11, 12]])
c_custom = a_custom @ b_custom

# NumPy implementation
a_np = np.array([[1, 2, 3], [4, 5, 6]])
b_np = np.array([[7, 8], [9, 10], [11, 12]])
c_np = a_np @ b_np

print("Your implementation:")
print(c_custom)
print("\nNumPy:")
print(c_np)
print("\n✓ Results match!" if c_custom.data == c_np.tolist() else "✗ Results don't match")

print("\n" + "="*60)
print("Key Differences:")
print("="*60)
print("1. Speed: NumPy uses optimized C code (100-1000x faster)")
print("2. Memory: NumPy uses contiguous memory (cache-friendly)")
print("3. Features: NumPy has broadcasting, fancy indexing, etc.")
print("4. Types: NumPy supports many numeric types (float32, int64, etc.)")
print("\nBut the LOGIC is the same! You now understand what's happening under the hood.")

## Summary

### What You've Learned

✅ **Magic methods** enable operator overloading in Python

✅ **Element-wise operations** work on corresponding elements

✅ **Matrix multiplication** computes dot products of rows and columns

✅ **NumPy internals** — you built a mini-version from scratch!

### Why This Matters

In the next labs, you'll:
- Build a `Value` class that tracks operations (computation graph)
- Use **NumPy** for efficient numeric computation (you've earned it!)
- Implement automatic differentiation for neural networks

Understanding these fundamentals will help you:
- Debug deep learning code more effectively
- Build custom operations when needed
- Understand how PyTorch and TensorFlow work internally

### Next Steps

In **Lab 2**, we'll introduce:
- Computation graphs for tracking operations
- The `Value` class that records its history
- NumPy for efficient numeric computation (finally!)
- Forward propagation through networks

**Great work!** 🎉 You've built the foundation for understanding deep learning frameworks.